In [ ]:
# normalizing the data and putting them into one image with multiple channels 
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from PIL import Image
import rasterio
from rasterio.plot import show
import tensorflow
from tensorflow.keras import layers, models
import os
from pprint import pprint

In [ ]:
def normalize_band(band, mask=None):
    """
    Normalize a 2D array to [0, 1] using min-max, only over valid pixels.
    
    Parameters:
        band: 2D NumPy array (e.g., from a TIFF)
        mask: Boolean array, same shape (True = valid pixel)
    
    Returns:
        Normalized array (same shape), NaNs or invalids set to 0
    """
    band = band.copy()
    if mask is None:
        mask = ~np.isnan(band)

    min_val = band[mask].min()
    max_val = band[mask].max()
    band[mask] = (band[mask] - min_val) / (max_val - min_val + 1e-8)
    band[~mask] = 0  # Optional: mask out invalids with 0
    
    return band


In [ ]:
ch4_plume = tiff.imread("./ghgsat2803/2022/11/28/C5/C5_23010168_20221128_20230531_LbOSaFE/C5_23010168_20221128_20230531_LbOSaFE_8061_CH4PL.tif")
ch4_error = tiff.imread("./ghgsat2803/2022/11/28/C5/C5_23010168_20221128_20230531_LbOSaFE/C5_23010168_20221128_20230531_LbOSaFE_CH4ER.tif")
albedo = tiff.imread("./ghgsat2803/2022/11/28/C5/C5_23010168_20221128_20230531_LbOSaFE/C5_23010168_20221128_20230531_LbOSaFE_ALB.tif")
flag = tiff.imread("./ghgsat2803/2022/11/28/C5/C5_23010168_20221128_20230531_LbOSaFE/C5_23010168_20221128_20230531_LbOSaFE_FLG.tif")

In [ ]:
# Load your tiffs and quality flag (FLG)
mask = valid_mask = (flag == 1) & (~np.isnan(ch4_plume))


ch4pl_norm = normalize_band(ch4_plume, mask)
ch4er_norm = normalize_band(ch4_error, mask)
alb_norm   = normalize_band(albedo, mask)
wind_speed = 1.7 # this speed was taken from the same dataset and the provided csv file with wind speed
wind_layer = np.ones_like(ch4pl_norm) * (wind_speed / 10.0)
true_emission_rate = 1561 # also gotten from the csv for testing 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

img = np.array(ch4pl_norm)  # or .jpg/.tif
plt.imshow(img)
plt.axis('off')  # optional: hides axis ticks
plt.show()



In [ ]:
stacked = np.stack([ch4pl_norm, ch4er_norm, alb_norm, wind_layer], axis=-1)

print("Stacked shape:", stacked.shape)  # Should print (H, W, 4)

In [ ]:
H, W, C = stacked.shape  # get height, width, channels

model = models.Sequential([
    layers.Input(shape=(H, W, C)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.GlobalAveragePooling2D(),  # aggregates spatial info
    layers.Dense(64, activation='relu'),
    layers.Dense(1)  # output: emission rate in kg/h
])

model.compile(optimizer='adam', loss='mse')

In [ ]:
x = np.expand_dims(stacked, axis=0)  # (1, H, W, 4)
y = np.array([[true_emission_rate]])  # (1, 1)

mmodel = model.fit(x, y, epochs=500)

In [ ]:


plt.plot(mmodel.history['loss'])
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.grid(True)
plt.show()


In [ ]:
def process_observation(obs_dir):
    import glob
    import tifffile as tiff
    import pandas as pd
    import numpy as np

    def normalize_band(band, mask):
        band = band.copy()
        valid = mask & ~np.isnan(band)
        band[valid] = (band[valid] - band[valid].min()) / (band[valid].max() - band[valid].min() + 1e-8)
        band[~valid] = 0
        return band

    def find_file(suffix):
        for f in os.listdir(obs_dir):
            if f.endswith(suffix):
                return os.path.join(obs_dir, f)
        return None

    ch4pl_path = find_file("CH4PL.tif")
    ch4er_path = find_file("CH4ER.tif")
    alb_path   = find_file("ALB.tif")
    flg_path   = find_file("FLG.tif")
    sr_csv_path = glob.glob(os.path.join(obs_dir, "*CH4SR.csv"))

    if not all([ch4pl_path, ch4er_path, alb_path, flg_path]) or not sr_csv_path:
        return None, None, f"Missing files in {obs_dir}"

    try:
        ch4pl = tiff.imread(ch4pl_path)
        ch4er = tiff.imread(ch4er_path)
        alb   = tiff.imread(alb_path)
        flg   = tiff.imread(flg_path)
    except Exception as e:
        return None, None, f"Error reading TIFFs: {e}"

    valid_mask = (flg == 1)
    ch4pl = normalize_band(ch4pl, valid_mask)
    ch4er = normalize_band(ch4er, valid_mask)
    alb   = normalize_band(alb, valid_mask)

    ch4pl[~valid_mask] = 0
    ch4er[~valid_mask] = 0
    alb[~valid_mask] = 0

    try:
        df = pd.read_csv(sr_csv_path[0])
        rate = df["Q-IME [kg/hr]"].values[0]
        wind = df["Wind speed [m/s]"].values[0] / 10.0
    except Exception as e:
        return None, None, f"Error reading metadata CSV: {e}"

    wind_layer = np.ones_like(ch4pl) * wind
    x = np.stack([ch4pl, ch4er, alb, wind_layer], axis=-1)
    y = rate
    return x, y, None


In [ ]:
input_data = []
labels = []
expected_shape = None

for dirpath, _, filenames in os.walk(root_dir):
    if any(f.endswith("CH4PL.tif") for f in filenames):
        x, y, error = process_observation(dirpath)
        if error:
            print(error)
            continue
        if expected_shape is None:
            expected_shape = x.shape
        if x.shape != expected_shape:
            print(f"Skipping {dirpath} due to shape mismatch: {x.shape} ≠ {expected_shape}")
            continue
        input_data.append(x)
        labels.append(y)

input_data = np.array(input_data)
labels = np.array(labels)

print(input_data, labels)


In [ ]:
from pathlib import Path
from collections import defaultdict
import re

SOURCE_DIR = Path(r"/Volumes/2tb_western/ghgsat data/ghgsat")

# Maps obs_id → {tag: [file1, file2, ...]}
grouped_files = defaultdict(lambda: defaultdict(list))

# Match common ID and data type (with optional numeric prefix like _582_)
pattern = re.compile(
    r"(?P<id>C[12]_\d+_\d{8}_\d{8}_[\w-]+?)_(\d+_)?(?P<tag>CH4CM|CH4PL|CH4SR|CH4|CH4ER|ALB|FLG|META|BRW)\.(tif|png|csv|json|pdf)"
)

for path in SOURCE_DIR.rglob("*"):
    if path.is_file():
        match = pattern.search(path.name)
        if match:
            obs_id = match.group("id")
            tag = match.group("tag")
            grouped_files[obs_id][tag].append(str(path))

# Optional: flatten single-item lists
for obs_id, tag_map in grouped_files.items():
    for tag, paths in tag_map.items():
        if len(paths) == 1:
            tag_map[tag] = paths[0]  # keep string for convenience

import json

# Recursively convert defaultdicts to plain dicts
def to_serializable(obj):
    if isinstance(obj, defaultdict):
        obj = {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, dict):
        obj = {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        obj = [to_serializable(v) for v in obj]
    return obj

# Save to JSON
with open("grouped_files.json", "w") as f:
    json.dump(to_serializable(grouped_files), f, indent=2)



In [ ]:
# Complete pipeline for loading, validating, resizing, and saving GHGSat plume data

import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
import json

# 1. Load grouped_files from a JSON file
def load_grouped_files(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

# 2. Load and resize a single-band .tif
def load_and_resize(path, shape=(128, 128)):
    try:
        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)
            if img.ndim != 2:
                return None
            return resize(img, shape, anti_aliasing=True)
    except Exception:
        return None

# 3. Build the dataset from grouped_files
def build_resized_dataset(grouped_files, target_shape=(128, 128)):
    X = []
    y = []
    stats = Counter()

    for obs_id, files in grouped_files.items():
        required = ["CH4PL", "CH4SR", "CH4", "ALB", "FLG", "CH4ER"]
        if not all(k in files for k in required):
            stats["missing_required_keys"] += 1
            continue

        ch4pl_list = files["CH4PL"] if isinstance(files["CH4PL"], list) else [files["CH4PL"]]
        ch4sr_list = files["CH4SR"] if isinstance(files["CH4SR"], list) else [files["CH4SR"]]

        if len(ch4pl_list) != len(ch4sr_list):
            stats["mismatched_ch4pl_ch4sr"] += 1
            continue

        for ch4pl_path, ch4sr_path in zip(ch4pl_list, ch4sr_list):
            try:
                ch4pl = load_and_resize(ch4pl_path, shape=target_shape)
                ch4 = load_and_resize(files["CH4"], shape=target_shape)
                alb = load_and_resize(files["ALB"], shape=target_shape)
                flg = load_and_resize(files["FLG"], shape=target_shape)
                ch4er = load_and_resize(files["CH4ER"], shape=target_shape)

                if any(x is None for x in (ch4pl, ch4, alb, flg, ch4er)):
                    stats["bad_resize_or_load"] += 1
                    continue
                
                df = pd.read_csv(ch4sr_path)
                if "Q-IME [kg/hr]" not in df.columns:
                    stats["missing_tph_column"] += 1
                    continue
                target = df["Q-IME [kg/hr]"].values[0]  # in kg/hr

                img = np.stack([ch4pl, ch4, alb, flg, ch4er], axis=-1)
                X.append(img)
                y.append(target)
                stats["valid"] += 1

            except Exception:
                stats["exception"] += 1

    return np.array(X), np.array(y), stats

# 4. Save dataset to disk
def save_dataset(X, y, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    np.save(output_dir / "X_train.npy", X_train)
    np.save(output_dir / "X_test.npy", X_test)
    np.save(output_dir / "y_train.npy", y_train)
    np.save(output_dir / "y_test.npy", y_test)

    return {
        "X_train": X_train.shape,
        "X_test": X_test.shape,
        "y_train": y_train.shape,
        "y_test": y_test.shape
    }

# Entry point
def full_pipeline(json_path, output_dir="GHGsat", target_shape=(128, 128)):
    grouped_files = load_grouped_files(json_path)
    X, y, stats = build_resized_dataset(grouped_files, target_shape=target_shape)
    shapes = save_dataset(X, y, output_dir)
    return {"dataset_shapes": shapes, "load_stats": stats}



In [ ]:
results = full_pipeline("grouped_files.json")
print(results)


In [ ]:
import pandas as pd

# Pick any CH4SR CSV path from your grouped_files.json manually
csv_path = "/Volumes/2tb_western/.../some_CH4SR.csv"  # Replace this

df = pd.read_csv(csv_path)
print(df.columns)
print(df.head())



In [ ]:
from collections import Counter

def build_resized_dataset_debug(grouped_files, target_shape=(128, 128)):
    for obs_id, files in grouped_files.items():
        required = ["CH4PL", "CH4SR", "CH4", "ALB", "FLG", "CH4ER"]
        if not all(k in files for k in required):
            continue

        ch4pl_list = files["CH4PL"] if isinstance(files["CH4PL"], list) else [files["CH4PL"]]
        ch4sr_list = files["CH4SR"] if isinstance(files["CH4SR"], list) else [files["CH4SR"]]

        for ch4pl_path, ch4sr_path in zip(ch4pl_list, ch4sr_list):
            try:
                ch4pl = load_and_resize(ch4pl_path, shape=target_shape)
                ch4 = load_and_resize(files["CH4"], shape=target_shape)
                alb = load_and_resize(files["ALB"], shape=target_shape)
                flg = load_and_resize(files["FLG"], shape=target_shape)
                ch4er = load_and_resize(files["CH4ER"], shape=target_shape)

                # Check all images loaded
                if None in (ch4pl, ch4, alb, flg, ch4er):
                    raise ValueError("One or more .tif files failed to load or resize")

                # Try loading the target
                df = pd.read_csv(ch4sr_path)
                if "source_rate_tph" not in df.columns:
                    raise ValueError("Missing 'source_rate_tph' column in CSV")

                target = float(df["source_rate_tph"].values[0])
                img = np.stack([ch4pl, ch4, alb, flg, ch4er], axis=-1)

            except Exception as e:
                print(f"[{obs_id}]")
                print(f"  CH4PL: {ch4pl_path}")
                print(f"  CH4SR: {ch4sr_path}")
                print(f"  Error: {e}")
                return


# Load and run
grouped_files = load_grouped_files("grouped_files.json")
build_resized_dataset_debug(grouped_files)
